# Environment Setup Workbench

Use this notebook to create or refresh the `tatva` environment, install JAX and the local package, and verify that the runtime is ready.

This notebook is designed to work both on the local repository and on a copied repository such as the T7 drive.


## 1. Path Detection and Helpers

Run this cell first. It tries to auto-detect the repository root and defines a small command runner.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path


def detect_repo_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent])
    candidates.extend([
        Path('/Users/gausschang/Documents/University/Master/tatva'),
        Path('/Volumes/Gauss-T7/tatva'),
    ])
    for root in candidates:
        if (root / 'Velocity-weakening' / 'src' / 'velocity_weakening_tatva.py').exists() and (root / 'tatva' / 'legacy_velocity_weakening.py').exists():
            return root
    raise FileNotFoundError('Could not auto-detect the tatva repository root. Set REPO_ROOT manually in this cell.')


REPO_ROOT = detect_repo_root()
SRC_ROOT = REPO_ROOT / 'Velocity-weakening' / 'src'
CONDA_EXE = shutil.which('conda')

if CONDA_EXE is None:
    raise RuntimeError('`conda` was not found on PATH. Install Conda/Miniconda first or open Jupyter from a shell where `conda` is available.')


def run(cmd: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    print('$', ' '.join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd or REPO_ROOT),
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {cmd}')
    return completed


print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT  =', SRC_ROOT)
print('CONDA_EXE =', CONDA_EXE)


REPO_ROOT = /Volumes/Gauss-T7/tatva
SRC_ROOT  = /Volumes/Gauss-T7/tatva/Velocity-weakening/src
CONDA_EXE = /opt/homebrew/bin/conda


## 2. Setup Configuration

Edit these values if needed. For a standard CPU setup, the defaults are usually fine.


In [2]:
ENV_NAME = 'tatva'
PYTHON_VERSION = '3.12'
CREATE_ENV_IF_MISSING = True
FORCE_PIP_UPGRADE = True
INSTALL_EDITABLE_TATVA = True
INSTALL_CPU_JAX = True
INSTALL_OPTIONAL_NOTEBOOK_PACKAGES = True
REGISTER_IPYKERNEL = True

# Keep GPU plugin optional. CPU JAX is the default stable choice for this project.
INSTALL_JAX_METAL = False

NOTEBOOK_PACKAGES = [
    'matplotlib',
    'h5py',
    'pytest',
    'ipykernel',
    'pandas',
]

print(json.dumps({
    'ENV_NAME': ENV_NAME,
    'PYTHON_VERSION': PYTHON_VERSION,
    'CREATE_ENV_IF_MISSING': CREATE_ENV_IF_MISSING,
    'INSTALL_CPU_JAX': INSTALL_CPU_JAX,
    'INSTALL_JAX_METAL': INSTALL_JAX_METAL,
    'INSTALL_EDITABLE_TATVA': INSTALL_EDITABLE_TATVA,
}, indent=2))


{
  "ENV_NAME": "tatva",
  "PYTHON_VERSION": "3.12",
  "CREATE_ENV_IF_MISSING": true,
  "INSTALL_CPU_JAX": true,
  "INSTALL_JAX_METAL": false,
  "INSTALL_EDITABLE_TATVA": true
}


## 3. Create the Conda Environment If Needed

This cell checks whether the named environment exists. If not, it creates it.


In [3]:
env_list = run([CONDA_EXE, 'env', 'list', '--json'])
envs = json.loads(env_list.stdout).get('envs', [])
env_exists = any(Path(path).name == ENV_NAME for path in envs)
print('env_exists =', env_exists)

if not env_exists:
    if not CREATE_ENV_IF_MISSING:
        raise RuntimeError(f'Conda environment `{ENV_NAME}` does not exist and CREATE_ENV_IF_MISSING is False.')
    run([CONDA_EXE, 'create', '-n', ENV_NAME, f'python={PYTHON_VERSION}', '-y'])
else:
    print(f'Conda environment `{ENV_NAME}` already exists.')


$ /opt/homebrew/bin/conda env list --json
{
  "GID": 20,
  "UID": 502,
  "active_prefix": "/opt/homebrew/Caskroom/miniconda/base/envs/tatva",
  "active_prefix_name": "tatva",
  "av_data_dir": "/opt/homebrew/Caskroom/miniconda/base/etc/conda",
  "av_metadata_url_base": null,
  "channels": [
    "https://repo.anaconda.com/pkgs/main/osx-arm64",
    "https://repo.anaconda.com/pkgs/main/noarch",
    "https://repo.anaconda.com/pkgs/r/osx-arm64",
    "https://repo.anaconda.com/pkgs/r/noarch"
  ],
  "conda_build_version": "not installed",
  "conda_env_version": "24.11.3",
  "conda_location": "/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/conda",
  "conda_prefix": "/opt/homebrew/Caskroom/miniconda/base",
  "conda_shlvl": 1,
  "conda_version": "24.11.3",
  "config_files": [
    "/opt/homebrew/Caskroom/miniconda/base/.condarc",
    "/Users/b09501028/.condarc"
  ],
  "default_prefix": "/opt/homebrew/Caskroom/miniconda/base/envs/tatva",
  "env_vars": {
    "CIO_TEST": "<not set

## 4. Install or Refresh Packages

This cell upgrades pip tooling, installs CPU JAX by default, installs the local `tatva` package in editable mode, and adds notebook utilities.


In [4]:
def conda_run_python(args: list[str]) -> subprocess.CompletedProcess[str]:
    return run([CONDA_EXE, 'run', '-n', ENV_NAME, 'python', *args])


if FORCE_PIP_UPGRADE:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

if INSTALL_CPU_JAX:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'jax', 'jaxlib'])

if INSTALL_JAX_METAL:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', 'jax-metal'])

if INSTALL_EDITABLE_TATVA:
    conda_run_python(['-m', 'pip', 'install', '-e', str(REPO_ROOT)])

if INSTALL_OPTIONAL_NOTEBOOK_PACKAGES and NOTEBOOK_PACKAGES:
    conda_run_python(['-m', 'pip', 'install', '--upgrade', *NOTEBOOK_PACKAGES])

if REGISTER_IPYKERNEL:
    conda_run_python(['-m', 'ipykernel', 'install', '--user', '--name', ENV_NAME, '--display-name', f'Python ({ENV_NAME})'])


$ /opt/homebrew/bin/conda run -n tatva python -m pip install --upgrade pip setuptools wheel


$ /opt/homebrew/bin/conda run -n tatva python -m pip install --upgrade jax jaxlib
  Using cached scipy-1.17.1-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.1 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1.6/3.1 MB 14.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/58.8 MB ? eta -:--:--
   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/58.8 MB 12.2 MB/s eta 0:00:05
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/58.8 MB 9.2 MB/s eta 0:00:07
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/58.8 MB 8.5 MB/s eta 0:00:07
   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/58.8 MB 8.2 MB/s eta 0:00:07
   ━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/58.8 MB 7.8 MB/s eta 0:00:07
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/58.8 MB 7.1 MB/s eta 

## 5. Verify the Environment

This cell checks the package versions, imports `tatva`, and prints the available JAX devices.


In [5]:
verify_script = r'''
import json
import importlib.metadata as md
from pathlib import Path

import jax
import h5py
import matplotlib
import tatva

root = Path(r"''' + str(REPO_ROOT) + '''")
payload = {
    'repo_root': str(root),
    'jax_version': md.version('jax'),
    'jaxlib_version': md.version('jaxlib'),
    'matplotlib_version': md.version('matplotlib'),
    'h5py_version': md.version('h5py'),
    'tatva_import_ok': hasattr(tatva, '__file__'),
    'tatva_module': str(getattr(tatva, '__file__', 'unknown')),
    'backend': jax.default_backend(),
    'devices': [str(d) for d in jax.devices()],
}
print(json.dumps(payload, indent=2))
'''

conda_run_python(['-c', verify_script])


$ /opt/homebrew/bin/conda run -n tatva python -c 
import json
import importlib.metadata as md
from pathlib import Path

import jax
import h5py
import matplotlib
import tatva

root = Path(r"/Volumes/Gauss-T7/tatva")
payload = {
    'repo_root': str(root),
    'jax_version': md.version('jax'),
    'jaxlib_version': md.version('jaxlib'),
    'matplotlib_version': md.version('matplotlib'),
    'h5py_version': md.version('h5py'),
    'tatva_import_ok': hasattr(tatva, '__file__'),
    'tatva_module': str(getattr(tatva, '__file__', 'unknown')),
    'backend': jax.default_backend(),
    'devices': [str(d) for d in jax.devices()],
}
print(json.dumps(payload, indent=2))

{
  "repo_root": "/Volumes/Gauss-T7/tatva",
  "jax_version": "0.9.2",
  "jaxlib_version": "0.9.2",
  "matplotlib_version": "3.10.8",
  "h5py_version": "3.16.0",
  "tatva_import_ok": true,
  "tatva_module": "/Volumes/Gauss-T7/tatva/tatva/__init__.py",
  "backend": "cpu",
  "devices": [
    "TFRT_CPU_0"
  ]
}




CompletedProcess(args=['/opt/homebrew/bin/conda', 'run', '-n', 'tatva', 'python', '-c', '\nimport json\nimport importlib.metadata as md\nfrom pathlib import Path\n\nimport jax\nimport h5py\nimport matplotlib\nimport tatva\n\nroot = Path(r"/Volumes/Gauss-T7/tatva")\npayload = {\n    \'repo_root\': str(root),\n    \'jax_version\': md.version(\'jax\'),\n    \'jaxlib_version\': md.version(\'jaxlib\'),\n    \'matplotlib_version\': md.version(\'matplotlib\'),\n    \'h5py_version\': md.version(\'h5py\'),\n    \'tatva_import_ok\': hasattr(tatva, \'__file__\'),\n    \'tatva_module\': str(getattr(tatva, \'__file__\', \'unknown\')),\n    \'backend\': jax.default_backend(),\n    \'devices\': [str(d) for d in jax.devices()],\n}\nprint(json.dumps(payload, indent=2))\n'], returncode=0, stdout='{\n  "repo_root": "/Volumes/Gauss-T7/tatva",\n  "jax_version": "0.9.2",\n  "jaxlib_version": "0.9.2",\n  "matplotlib_version": "3.10.8",\n  "h5py_version": "3.16.0",\n  "tatva_import_ok": true,\n  "tatva_module

## 6. Optional Smoke Test

Turn this on if you want to run the repository smoke tests immediately after setup.


In [6]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    run([
        CONDA_EXE,
        'run',
        '-n',
        ENV_NAME,
        'pytest',
        str(REPO_ROOT / 'tests' / 'test_legacy_velocity_weakening.py'),
        '-q',
    ])
else:
    print('Set RUN_SMOKE_TEST = True if you want to run pytest now.')


Set RUN_SMOKE_TEST = True if you want to run pytest now.
